In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [ ]:
from Bio import SeqIO

file_path = our_data + "Hypericum_PF00067_Merge97.pep"
records = list(SeqIO.parse(file_path, "fasta"))

data = {"enzyme": [], "Sequence": []}


for record in records:
    data["enzyme"].append(record.id)
    data["Sequence"].append(str(record.seq))

df_enzymedata = pd.DataFrame(data)

In [4]:
df_enzymedata

,enzyme,Sequence
0,Hste_113815,MNLLILLLLSLILVLVFYFTYSLIWVPLRVQAHFRAQGVTGPGFRP...
1,Hlea_018408,MPDTIQCLNPFPSFATTNIYKPTDKKTMDQFYLTLLLLFVSSIALS...
2,Hflo_189074,MDGLYLKLALTLLLSIFIAYKLFFRKKHNYPPGPRALPIIGNLHLI...
3,Hflo_160077,MIMSPFVLYSFVVAVLVYFLVSLKSALSGRRLPPGPKPWPIVGNLP...
4,Hlea_230320,MVHSITKIEQEMLSASVFLAGATAIALILLFFWNSKRSSDKTSRLP...
...,...,...
321,Hflo_298643,MGFFHLLSLAIVGVFLYGFFKFLISCWVSPTRAYLKLRRNGFGGPT...
322,Hflo_264086,MDVSTALMLVAGVTAYLLWFTFISRTLRGPRVWPLLGSLPGLIENC...
323,Hflo_057832,MDVSTALMLVAGVTAYLLWFTFISRTLRGPRVWPLLGSLPGLIENC...
324,Hflo_121886,MELVTIFTLIVCVLTLLTIIRTISRSSYKNGGAKLPPGPSPLPVLG...


In [ ]:
data = {
    "substrate1": ["1,3,5-trihydroxyxanthone"],
    "substrate2": ["1,3,7-trihydroxyxanthone"],
}

df_substratedata = pd.DataFrame(data)
df_substratedata

,substrate1,substrate2
0,"1,3,5-trihydroxyxanthone","1,3,7-trihydroxyxanthone"


In [ ]:
df_enzymedata["key"] = 1
df_substratedata["key"] = 1

result = pd.merge(df_enzymedata, df_substratedata, on="key", how="outer")
result.drop(columns="key", inplace=True)

In [7]:
result

,enzyme,Sequence,substrate1,substrate2
0,Hste_113815,MNLLILLLLSLILVLVFYFTYSLIWVPLRVQAHFRAQGVTGPGFRP...,"1,3,5-trihydroxyxanthone","1,3,7-trihydroxyxanthone"
1,Hlea_018408,MPDTIQCLNPFPSFATTNIYKPTDKKTMDQFYLTLLLLFVSSIALS...,"1,3,5-trihydroxyxanthone","1,3,7-trihydroxyxanthone"
2,Hflo_189074,MDGLYLKLALTLLLSIFIAYKLFFRKKHNYPPGPRALPIIGNLHLI...,"1,3,5-trihydroxyxanthone","1,3,7-trihydroxyxanthone"
3,Hflo_160077,MIMSPFVLYSFVVAVLVYFLVSLKSALSGRRLPPGPKPWPIVGNLP...,"1,3,5-trihydroxyxanthone","1,3,7-trihydroxyxanthone"
4,Hlea_230320,MVHSITKIEQEMLSASVFLAGATAIALILLFFWNSKRSSDKTSRLP...,"1,3,5-trihydroxyxanthone","1,3,7-trihydroxyxanthone"
...,...,...,...,...
321,Hflo_298643,MGFFHLLSLAIVGVFLYGFFKFLISCWVSPTRAYLKLRRNGFGGPT...,"1,3,5-trihydroxyxanthone","1,3,7-trihydroxyxanthone"
322,Hflo_264086,MDVSTALMLVAGVTAYLLWFTFISRTLRGPRVWPLLGSLPGLIENC...,"1,3,5-trihydroxyxanthone","1,3,7-trihydroxyxanthone"
323,Hflo_057832,MDVSTALMLVAGVTAYLLWFTFISRTLRGPRVWPLLGSLPGLIENC...,"1,3,5-trihydroxyxanthone","1,3,7-trihydroxyxanthone"
324,Hflo_121886,MELVTIFTLIVCVLTLLTIIRTISRSSYKNGGAKLPPGPSPLPVLG...,"1,3,5-trihydroxyxanthone","1,3,7-trihydroxyxanthone"
